## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog. 

<b>Caution</b> ... This notebook requires significant execution time as there are 9319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer and typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and use of cloud computing services. 

### Load Python Dependencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

<h3>Extracting Landsat Data Using API Calls</h3> <p align="justify"> The API-based method allows us to efficiently access <b>Landsat</b> data for specific coordinates and time periods, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can query individual bands or compute indices like <b>NDMI</b> on-the-fly. This approach reduces storage requirements and simplifies data preprocessing, making it ideal for large-scale environmental and water quality analysis. </p>

<p>The <b>compute_Landsat_values</b> function extracts Landsat surface reflectance values for specific sampling locations using a 100 m focal buffer around each point. For each location:</p>

<ul>
  <li>A bounding box (bbox) is created around the latitude and longitude coordinates.</li>
  <li>The Microsoft Planetary Computer API is queried for Landsat-8 Level-2 surface reflectance imagery within the date range.</li>
  <li>The nearest low-cloud (<10% cloud cover) scene is selected, and the specified bands (<b>green</b>, <b>nir08</b>, <b>swir16</b>, <b>swir22</b>) are loaded.</li>
  <li>Median values of the pixels within the bounding box are computed to reduce the effect of noise or outliers.</li>
</ul>

<p><b>Why the buffer value is 0.00089831:</b></p>

<p>We want a ~100 m buffer around each point. At the equator, 1 degree ≈ 110 km. Therefore, the degree equivalent of 100 m is:</p>

<p style="text-align:center;">
  <em>buffer_deg = 100 m / 110,000 m/deg ≈ 0.00089831</em>
</p>

<p>This slightly adjusted value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing a ~100 m area around each sampling location.</p>


In [ ]:
# Mapping: STAC asset name -> output column name
DESIRED_BANDS = {
    "coastal": "coastal", "blue": "blue", "green": "green", "red": "red",
    "nir08": "nir", "swir16": "swir16", "swir22": "swir22",
    "lwir": "lwir", "lwir11": "lwir11",
    "trad": "trad", "urad": "urad", "drad": "drad",
    "atran": "atran", "emis": "emis", "emsd": "emsd", "cdist": "cdist",
}

ZERO_IS_NODATA = {"coastal", "blue", "green", "red", "nir08", "swir16", "swir22", "lwir", "lwir11"}

ALL_OUTPUT_COLUMNS = list(dict.fromkeys(DESIRED_BANDS.values())) + ["scene_date", "scene_dist_km"]
MAX_DAYS_DISTANCE = 180

tqdm.pandas()

import time
import threading
import math

_catalog = None
_auth_lock = threading.Lock()
_request_count = 0

def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def _get_catalog():
    """Get STAC catalog WITHOUT sign_inplace — items stay unsigned."""
    global _catalog
    if _catalog is None:
        with _auth_lock:
            if _catalog is None:
                _catalog = pystac_client.Client.open(
                    "https://planetarycomputer.microsoft.com/api/stac/v1",
                )
    return _catalog

def _force_refresh_auth(reason="auth error"):
    """Clear the SAS token cache so next pc.sign() gets a fresh token."""
    with _auth_lock:
        token_info = ""
        for url, tok in pc.sas.TOKEN_CACHE.items():
            container = url.split("/")[-1] if "/" in url else url
            token_info = f" (container={container}, ttl={tok.ttl():.0f}s, reqs={_request_count})"
        pc.sas.TOKEN_CACHE.clear()
        print(f"  [AUTH REFRESH] {reason}{token_info} — cleared TOKEN_CACHE")

def _is_auth_error(e):
    msg = str(e).lower()
    return "authentication" in msg or "403" in msg or "signature" in msg or "authorization" in msg

def _run_with_retries(operation, max_retries=2):
    for attempt in range(1, max_retries + 1):
        try:
            return operation()
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(2)

def _process_row(row, nan_result):
    global _request_count
    lat = row['Latitude']
    lon = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    if pd.isna(date):
        return nan_result

    # 100m bbox — matches original code
    bbox_size = 0.00089831
    bbox = [lon - bbox_size/2, lat - bbox_size/2, lon + bbox_size/2, lat + bbox_size/2]

    # Search with the 100m bbox and full date range (like original code)
    catalog = _get_catalog()
    search = catalog.search(
        collections=["landsat-c2-l2"],
        bbox=bbox,
        datetime="2011-01-01/2015-12-31",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    items = _run_with_retries(lambda: search.item_collection())

    if not items:
        return nan_result

    sample_date_utc = date.tz_localize("UTC") if date.tzinfo is None else date.tz_convert("UTC")
    items_sorted = sorted(
        items,
        key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc)
    )
    selected_item = items_sorted[0]

    scene_dt = pd.to_datetime(selected_item.properties["datetime"]).tz_convert("UTC")
    days_diff = abs((scene_dt - sample_date_utc).days)
    if days_diff > MAX_DAYS_DISTANCE:
        return nan_result

    # Compute distance from sample point to scene centroid
    geom = selected_item.geometry
    coords = geom["coordinates"][0]  # exterior ring of polygon
    scene_lon = sum(c[0] for c in coords) / len(coords)
    scene_lat = sum(c[1] for c in coords) / len(coords)
    dist_km = _haversine_km(lat, lon, scene_lat, scene_lon)

    available_assets = set(selected_item.assets.keys())
    bands_to_load = [b for b in DESIRED_BANDS if b in available_assets]
    if not bands_to_load:
        return nan_result

    _request_count += 1
    data = _run_with_retries(
        lambda: stac_load([pc.sign(selected_item)], bands=bands_to_load, bbox=bbox).isel(time=0)
    )

    results = {col: np.nan for col in ALL_OUTPUT_COLUMNS}
    results["scene_date"] = scene_dt.strftime("%Y-%m-%d")
    results["scene_dist_km"] = round(dist_km, 2)

    for stac_name, output_name in DESIRED_BANDS.items():
        if stac_name in data:
            val = float(data[stac_name].astype("float").median(skipna=True).values)
            if stac_name in ZERO_IS_NODATA and val == 0:
                results[output_name] = np.nan
            elif stac_name not in ZERO_IS_NODATA and val == -9999:
                results[output_name] = np.nan
            else:
                results[output_name] = val

    ok_bands = sum(1 for k, v in results.items() if k not in ("scene_date", "scene_dist_km") and pd.notna(v))
    print(f"  [OK] lat={lat:.2f} lon={lon:.2f} → {ok_bands} bands, scene={results['scene_date']}, dist={dist_km:.1f}km (req #{_request_count})")
    return pd.Series(results)

def compute_Landsat_values(row):
    lat = row['Latitude']
    lon = row['Longitude']
    nan_result = pd.Series({col: np.nan for col in ALL_OUTPUT_COLUMNS})

    try:
        return _process_row(row, nan_result)
    except Exception as e:
        if _is_auth_error(e):
            _force_refresh_auth()
            time.sleep(5)
            try:
                return _process_row(row, nan_result)
            except Exception as e2:
                print(f"  [FAILED] lat={lat:.2f} lon={lon:.2f}: {type(e2).__name__} (after auth refresh)")
                return nan_result
        else:
            print(f"  [FAILED] lat={lat:.2f} lon={lon:.2f}: {type(e).__name__}")
            return nan_result

### Extracting features for the training dataset

In [3]:
Water_Quality_df=pd.read_csv('water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [4]:
Water_Quality_df.shape

(9319, 6)

<h3>Note:</h3>
<p>The Landsat data extraction process for all 9,319 locations typically requires 7+ hours when executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors, or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.</p>

<p>In this notebook, we provide a sample code snippet demonstrating how to extract data for the first 200 locations. Participants are encouraged to follow the same batching approach to extract data for all 9,319 locations safely and efficiently.</p>

<p>We have already executed the full extraction for all 9,319 locations and saved the output to <b>landsat_features_training.csv</b>, which will be used in the benchmark notebook.
Similarly, participants can extract Landsat features in batches, combine the batch outputs, and save the final merged dataset as <b>landsat_features_training.csv</b> to ensure the benchmark notebook runs smoothly.</p>

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

train_features_path = "landsat_features_training.csv"
batch_dir = "landsat_batches_train_v2"
batch_size = 200
NUM_WORKERS = 4
NAN_STREAK_THRESHOLD = 10  # trigger auth refresh after this many consecutive NaN completions

os.makedirs(batch_dir, exist_ok=True)

num_rows = len(Water_Quality_df)
num_batches = (num_rows + batch_size - 1) // batch_size

print(f"Total rows: {num_rows} | Batch size: {batch_size} | Batches: {num_batches} | Workers: {NUM_WORKERS}")

for batch_idx in range(num_batches):
    batch_path = os.path.join(batch_dir, f"batch_{batch_idx:04d}.csv")

    if os.path.exists(batch_path):
        print(f"Batch {batch_idx+1}/{num_batches} already exists, skipping.")
        continue

    start = batch_idx * batch_size
    end = min(start + batch_size, num_rows)
    batch_df = Water_Quality_df.iloc[start:end]

    print(f"\nBatch {batch_idx+1}/{num_batches} (rows {start}–{end-1})")

    results = [None] * len(batch_df)
    nan_streak = 0

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        future_to_idx = {
            executor.submit(compute_Landsat_values, row): i
            for i, (_, row) in enumerate(batch_df.iterrows())
        }
        done_count = 0
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            result = future.result()
            results[idx] = result
            done_count += 1

            # Track consecutive NaN completions (in completion order)
            if result.isna().all():
                nan_streak += 1
                if nan_streak == NAN_STREAK_THRESHOLD:
                    print(f"  [NAN STREAK] {NAN_STREAK_THRESHOLD} consecutive NaN — refreshing auth")
                    _force_refresh_auth(reason=f"NaN streak in batch {batch_idx}")
                    time.sleep(5)
                    nan_streak = 0  # reset so we can detect if it happens again
            else:
                nan_streak = 0

            if done_count % 50 == 0 or done_count == len(batch_df):
                print(f"  Progress: {done_count}/{len(batch_df)}")

    batch_features = pd.DataFrame(results)
    batch_features.to_csv(batch_path, index=False)
    nan_count = batch_features.isna().all(axis=1).sum()
    print(f"Saved {batch_path} — {nan_count}/{len(batch_features)} NaN ({nan_count/len(batch_features)*100:.0f}%)")

# Combine all batches
batch_files = sorted(
    [os.path.join(batch_dir, f) for f in os.listdir(batch_dir) if f.endswith(".csv")]
)
landsat_train_features = pd.concat([pd.read_csv(f) for f in batch_files], ignore_index=True)
landsat_train_features.to_csv(train_features_path, index=False)
print(f"\nCombined {len(batch_files)} batches → {train_features_path} ({len(landsat_train_features)} rows)")

<p><b>NDMI and MNDWI Indices:</b></p>
<p>In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:</p>
<ul>
  <li><b>NDMI (Normalized Difference Moisture Index):</b> Measures vegetation water content and surface moisture. Computed as <em>(NIR - SWIR16) / (NIR + SWIR16)</em>.</li>
  <li><b>MNDWI (Modified Normalized Difference Water Index):</b> Highlights open water features by enhancing water reflectance and suppressing built-up areas. Computed as <em>(Green - SWIR16) / (Green + SWIR16)</em>.</li>
</ul>

<p>An <b>epsilon value</b> (<em>eps = 1e-10</em>) is added in the denominators to avoid division by zero. These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.</p>


In [ ]:
# Create indices: NDMI and MNDWI (now computed on properly scaled reflectance values)
eps = 1e-10
landsat_train_features['NDMI'] = (landsat_train_features['nir'] - landsat_train_features['swir16']) / (landsat_train_features['nir'] + landsat_train_features['swir16'] + eps)
landsat_train_features['MNDWI'] = (landsat_train_features['green'] - landsat_train_features['swir16']) / (landsat_train_features['green'] + landsat_train_features['swir16'] + eps)

In [ ]:
landsat_train_features['Latitude'] = Water_Quality_df['Latitude']
landsat_train_features['Longitude'] = Water_Quality_df['Longitude']
landsat_train_features['Sample Date'] = Water_Quality_df['Sample Date']
landsat_train_features = landsat_train_features[['Latitude', 'Longitude', 'Sample Date'] + ALL_OUTPUT_COLUMNS + ['NDMI', 'MNDWI']]

In [9]:
landsat_train_features.to_csv(train_features_path, index=False)

In [10]:
# Preview File
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


### Extracting features for the validation dataset

In [11]:
Validation_df=pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [12]:
Validation_df.shape

(200, 6)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

val_features_path = "landsat_features_validation.csv"
NUM_WORKERS = 4

print("Running Landsat feature extraction for validation data...")

results = [None] * len(Validation_df)
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    future_to_idx = {
        executor.submit(compute_Landsat_values, row): i
        for i, (_, row) in enumerate(Validation_df.iterrows())
    }
    done_count = 0
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        results[idx] = future.result()
        done_count += 1
        if done_count % 50 == 0 or done_count == len(Validation_df):
            print(f"  Progress: {done_count}/{len(Validation_df)}")

landsat_val_features = pd.DataFrame(results)
landsat_val_features.to_csv(val_features_path, index=False)
nan_count = landsat_val_features.isna().all(axis=1).sum()
print(f"Done — {nan_count}/{len(landsat_val_features)} NaN ({nan_count/len(landsat_val_features)*100:.0f}%)")

In [ ]:
# Create indices: NDMI and MNDWI (now computed on properly scaled reflectance values)
eps = 1e-10
landsat_val_features['NDMI'] = (landsat_val_features['nir'] - landsat_val_features['swir16']) / (landsat_val_features['nir'] + landsat_val_features['swir16'] + eps)
landsat_val_features['MNDWI'] = (landsat_val_features['green'] - landsat_val_features['swir16']) / (landsat_val_features['green'] + landsat_val_features['swir16'] + eps)

In [ ]:
landsat_val_features['Latitude'] = Validation_df['Latitude']
landsat_val_features['Longitude'] = Validation_df['Longitude']
landsat_val_features['Sample Date'] = Validation_df['Sample Date']
landsat_val_features = landsat_val_features[['Latitude', 'Longitude', 'Sample Date'] + ALL_OUTPUT_COLUMNS + ['NDMI', 'MNDWI']]

In [16]:
landsat_val_features.to_csv(val_features_path, index=False)

In [17]:
# Preview File
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


In [ ]:
# Quick diagnostic: check actual asset keys
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

# Check the first 2 training data points
print("=== First 2 training data points ===")
for i in range(2):
    row = Water_Quality_df.iloc[i]
    lat, lon = row['Latitude'], row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True)
    bbox_size = 0.00089831
    bbox = [lon - bbox_size/2, lat - bbox_size/2, lon + bbox_size/2, lat + bbox_size/2]
    search = catalog.search(
        collections=["landsat-c2-l2"], bbox=bbox,
        datetime=f"{(date - pd.Timedelta(days=30)).strftime('%Y-%m-%d')}/{(date + pd.Timedelta(days=30)).strftime('%Y-%m-%d')}",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    items = search.item_collection()
    if items:
        items = sorted(items, key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - date.tz_localize("UTC")))
        item = items[0]
        print(f"\nRow {i}: {item.properties.get('platform')} | {item.properties.get('datetime')}")
        print(f"  Asset keys: {sorted(item.assets.keys())}")
        # Also show last item
        item_last = items[-1]
        print(f"  Last item: {item_last.properties.get('platform')} | {item_last.properties.get('datetime')}")
        print(f"  Asset keys: {sorted(item_last.assets.keys())}")

# Check the last 2 training data points
print("\n=== Last 2 training data points ===")
for i in range(len(Water_Quality_df) - 2, len(Water_Quality_df)):
    row = Water_Quality_df.iloc[i]
    lat, lon = row['Latitude'], row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True)
    bbox_size = 0.00089831
    bbox = [lon - bbox_size/2, lat - bbox_size/2, lon + bbox_size/2, lat + bbox_size/2]
    search = catalog.search(
        collections=["landsat-c2-l2"], bbox=bbox,
        datetime=f"{(date - pd.Timedelta(days=30)).strftime('%Y-%m-%d')}/{(date + pd.Timedelta(days=30)).strftime('%Y-%m-%d')}",
        query={"eo:cloud_cover": {"lt": 10}},
    )
    items = search.item_collection()
    if items:
        items = sorted(items, key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - date.tz_localize("UTC")))
        item = items[0]
        print(f"\nRow {i}: {item.properties.get('platform')} | {item.properties.get('datetime')}")
        print(f"  Asset keys: {sorted(item.assets.keys())}")
    else:
        print(f"\nRow {i}: No items found within ±30 days of {date.strftime('%Y-%m-%d')}")